# Bronze Layer - Raw Data Ingestion

## Check the volume

In [0]:
display(dbutils.fs.ls("/Volumes/taxicatalog/taxi_project-schema/taxivolume"))

## Define paths

In [0]:
volume_path = "/Volumes/taxicatalog/taxi_project-schema/taxivolume"

trip_path = f"{volume_path}/taxi_trip.csv"
location_path = f"{volume_path}/location.csv"

print(trip_path)
print(location_path)

## Read `taxi_trip.csv`

In [0]:
from pyspark.sql.functions import current_timestamp, col

taxi_trip_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(trip_path)
)

taxi_trip_bronze = (
    taxi_trip_bronze
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

display(taxi_trip_bronze)

## Check the schema

In [0]:
taxi_trip_bronze.printSchema()

## Read `location.csv`

In [0]:
from pyspark.sql.functions import current_timestamp, col

taxi_location_bronze = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(location_path)
)

taxi_location_bronze = (
    taxi_location_bronze
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

display(taxi_location_bronze)

## Check location schema

In [0]:
taxi_location_bronze.printSchema()

## Write the Bronze trip table

In [0]:
(
    taxi_trip_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("taxicatalog.`taxi_project-schema`.taxi_trip_bronze")
)

## Write the Bronze location table

In [0]:
(
    taxi_location_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("taxicatalog.`taxi_project-schema`.taxi_location_bronze")
)

## Verify tables

In [0]:
display(
    spark.table("taxicatalog.`taxi_project-schema`.taxi_trip_bronze")
)

In [0]:
display(
    spark.table("taxicatalog.`taxi_project-schema`.taxi_location_bronze")
)